# Diagnóstico de marca empleadora — DHL Supply Chain vs. peers
## Sentimiento (DistilBERT) + topic modeling (BERTopic) sobre reseñas de Glassdoor

**Encargo de RRHH:** identificar puntos fuertes y áreas de mejora de **DHL Supply Chain** como empleador a partir de Glassdoor, comparados con sus competidores directos.

**Peers:** UPS, FedEx, MAERSK, DB-Schenker y Kuehne+Nagel. Todos son operadores logísticos globales (paquetería, freight forwarding, contract logistics) que compiten por el mismo talento: operarios de almacén y transporte, supervisores de operaciones, planificadores y perfiles de supply chain. Todos superan las 300 reseñas.

| Sección | Contenido |
|---|---|
| 0 | Carga y preparación |
| 1 | EDA |
| 2 | Sentimiento en pros y cons (control de calidad) |
| 3 | Topic modeling con BERTopic |
| 4 | Comparación con peers (balance neto) |
| 5 | Empleados actuales vs. antiguos |
| 6 | Matriz de priorización y recomendaciones |

In [2]:
# Colab (Entorno > Cambiar tipo de entorno > GPU). transformers ya viene instalado; BERTopic no.
!pip install -q bertopic sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


---
## 0. Carga y preparación

In [3]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

TARGET = 'DHL-Supply-Chain'
# Solo la entidad 'DHL-Supply-Chain': hay otras (DHL, DHL-Global-Forwarding, Deutsche-Post-&-DHL)
# que son negocios distintos (paquetería/forwarding); mezclarlas diluiría el diagnóstico.
PEERS = ['UPS', 'FedEx', 'MAERSK', 'DB-Schenker', 'Kuehne+Nagel']
FIRMS = [TARGET] + PEERS

# Tope de 2.000 reseñas por empresa con semilla fija: UPS tiene ~17.500 y dominaría
# el modelo de topics conjunto; además acota el tiempo de DistilBERT/BERTopic.
MAX_POR_EMPRESA = 2000
SEED = 42

# Localización del dataset: carpeta actual o Google Drive (en Colab subir 286 MB por el panel es lento,
# por eso se admite tenerlo en "Mi unidad" y montar Drive).
import os
RUTA = 'glassdoor_reviews_hr.parquet'
if not os.path.exists(RUTA):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        RUTA = '/content/drive/MyDrive/glassdoor_reviews_hr.parquet'
    except ImportError:
        pass
assert os.path.exists(RUTA), f'No encuentro el dataset en {RUTA}: súbelo a la carpeta de trabajo o a Mi unidad de Google Drive'

raw = pd.read_parquet(RUTA, filters=[('firm', 'in', FIRMS)])
print('Reseñas disponibles por empresa:')
print(raw['firm'].value_counts())
print('\nNulos por columna:\n', raw.isna().sum())

# DECISIÓN: DHL-Supply-Chain solo tiene reseñas hasta junio 2021; los peers llegan a 2023.
# Recortamos todas las empresas a la misma ventana para que la comparación sea justa:
# si no, compararíamos a DHL (2018-2021) con peers que incluyen la crisis logística de 2022.
# Alternativa descartada: unir entidades DHL (paquetería/forwarding = otro negocio y otro perfil).
raw['date_review'] = pd.to_datetime(raw['date_review'])
FIN = raw.loc[raw['firm'] == TARGET, 'date_review'].max()
raw = raw[raw['date_review'] <= FIN]
print(f'\nVentana común: {raw.date_review.min():%Y-%m-%d} -> {FIN:%Y-%m-%d}')
print(raw['firm'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'glassdoor_reviews_hr.parquet'

In [ ]:
# Limpieza previa al muestreo
for col in ['pros', 'cons']:
    raw[col] = raw[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

n0 = len(raw)
raw = raw.drop_duplicates(subset=['firm', 'pros', 'cons'])           # duplicados exactos
raw = raw[(raw['pros'].str.len() >= 3) & (raw['cons'].str.len() >= 3)]  # textos vacíos o 'nan'
print(f'Eliminadas {n0 - len(raw):,} filas duplicadas o sin texto')

df = pd.concat([g.sample(n=min(MAX_POR_EMPRESA, len(g)), random_state=SEED)
                for _, g in raw.groupby('firm')], ignore_index=True)
print(f'Muestra de trabajo: {len(df):,} reseñas')
print(df['firm'].value_counts())

La columna `current` mezcla situación y antigüedad (`"Former Employee, more than 3 years"`). La separo en dos variables y traduzco `recommend` (v = sí, o = sin opinión, x = no).

In [ ]:
def situacion(s):
    s = str(s).lower()
    if s.startswith('current'): return 'Actual'
    if s.startswith('former'):  return 'Antiguo'
    return 'Desconocido'

def antiguedad(s):
    m = re.search(r'(less|more) than (\d+) year', str(s))
    return f'{m.group(1)} than {m.group(2)}y' if m else 'n/d'

df['situacion']  = df['current'].map(situacion)
df['antiguedad'] = df['current'].map(antiguedad)
df['recomienda'] = df['recommend'].map({'v': 'Sí', 'o': 'Neutral', 'x': 'No'})
df['date_review'] = pd.to_datetime(df['date_review'])
df['anio'] = df['date_review'].dt.year
df['es_target'] = np.where(df['firm'] == TARGET, TARGET, 'Peers')

print(df['situacion'].value_counts(), '\n')
print(df['antiguedad'].value_counts())
df[['firm','date_review','situacion','antiguedad','overall_rating','recomienda','pros','cons']].head()

---
## 1. EDA
### 1.1 Resumen por empresa

In [ ]:
resumen = (df.groupby('firm')
    .agg(n_resenas=('firm', 'size'),
         rating_medio=('overall_rating', 'mean'),
         pct_recomienda=('recommend', lambda s: (s == 'v').mean() * 100),
         pct_no_recomienda=('recommend', lambda s: (s == 'x').mean() * 100),
         pct_actuales=('situacion', lambda s: (s == 'Actual').mean() * 100))
    .round(2).sort_values('rating_medio', ascending=False))
# % recomienda calculado sobre el total (incluye 'sin opinión'), igual para todas las firmas
resumen

In [ ]:
orden = resumen.index.tolist()
colores = ['#D40511' if f == TARGET else '#9aa5b1' for f in orden]   # rojo DHL para destacar
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col, tit in zip(axes, ['rating_medio', 'pct_recomienda', 'pct_actuales'],
                        ['Rating medio (1-5)', '% recomienda', '% empleados actuales']):
    ax.barh(orden, resumen[col], color=colores)
    ax.set_title(tit); ax.invert_yaxis()
    for i, v in enumerate(resumen[col]): ax.text(v, i, f' {v:.1f}', va='center')
plt.tight_layout(); plt.show()

In [ ]:
# Distribución del rating: ¿la media esconde polarización?
dist = pd.crosstab(df['firm'], df['overall_rating'], normalize='index').mul(100).round(1).loc[orden]
dist.plot(kind='barh', stacked=True, figsize=(12, 4.5), colormap='RdYlGn')
plt.title('Distribución del overall_rating (%)'); plt.xlabel('%'); plt.gca().invert_yaxis()
plt.legend(title='Rating', bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()
dist

### 1.2 Empleados actuales vs. antiguos

In [ ]:
act_ant = df.pivot_table(index='firm', columns='situacion', values='overall_rating', aggfunc='mean').loc[orden]
act_ant['gap'] = act_ant['Actual'] - act_ant['Antiguo']
act_ant.round(2)

In [ ]:
act_ant[['Actual', 'Antiguo']].plot(kind='bar', figsize=(10, 4), color=['#FFCC00', '#D40511'])
plt.title('Rating medio: empleados actuales vs. antiguos'); plt.ylabel('Rating'); plt.ylim(1, 5)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# Rating por antigüedad (solo reseñas que la informan)
ant_ord = ['less than 1y', 'more than 1y', 'more than 3y', 'more than 5y', 'more than 8y', 'more than 10y']
antig = (df[df['antiguedad'] != 'n/d']
         .pivot_table(index='antiguedad', columns='es_target', values='overall_rating', aggfunc='mean')
         .reindex(ant_ord))
antig.plot(marker='o', figsize=(8, 4), color=['#D40511', '#9aa5b1'])
plt.title('Rating medio por antigüedad'); plt.ylabel('Rating'); plt.tight_layout(); plt.show()
antig.round(2)

### 1.3 Evolución del rating

In [ ]:
df['trimestre'] = df['date_review'].dt.to_period('Q').dt.to_timestamp()
evo = df.groupby(['trimestre', 'es_target'])['overall_rating'].agg(['mean', 'size']).reset_index()
# Trimestres con menos de 30 reseñas son ruido: se excluyen del gráfico
evo = evo[evo['size'] >= 30]

fig, ax = plt.subplots(figsize=(12, 4.5))
for grupo, c in [(TARGET, '#D40511'), ('Peers', '#9aa5b1')]:
    d = evo[evo['es_target'] == grupo]
    ax.plot(d['trimestre'], d['mean'], marker='o', label=grupo, color=c)
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-31'), alpha=0.1, color='orange', label='COVID / boom e-commerce')
ax.set_title('Evolución trimestral del rating medio'); ax.set_ylabel('Rating'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
por_anio = df.pivot_table(index='anio', columns='firm', values='overall_rating', aggfunc='mean')[orden]
print('Nº de reseñas por año:'); print(pd.crosstab(df['anio'], df['firm'])[orden], '\n')
por_anio.round(2)

### Conclusiones del EDA (ventana común jul-2018 → jun-2021)
- **Posición media-baja:** DHL Supply Chain (3,52) es 4ª de 6, empatada con Kuehne+Nagel y por delante solo de UPS y DB-Schenker. MAERSK (3,98) es el *benchmark* del sector. **Acción:** estudiar qué hace MAERSK distinto (secciones 3-4).
- **Polarización:** un 20 % de reseñas de DHL son de 1-2 estrellas (MAERSK ~10 %, FedEx ~15 %). Hay un grupo de empleados muy descontento → hay que identificar sus quejas en los cons.
- **Recomendación:** solo el 44 % recomienda DHL y un 26 % dice explícitamente que no (vs 18 % en FedEx). La reputación boca a boca afecta directamente a la atracción de talento.
- **Antigüedad:** DHL puntúa por debajo de los peers en todos los tramos de 1 a 8 años y solo supera a los peers con >10 años. **Acción:** el riesgo de fuga está en los perfiles de antigüedad media; hipótesis a validar con topics: carrera/promoción.
- **Actuales vs antiguos:** el gap (0,41) está en la media del sector → la pérdida de satisfacción al salir no es anómala.
- **Evolución:** DHL mejora en 2020 (3,66, pico del boom e-commerce/COVID) y retrocede en 2021 (3,51) mientras Kuehne+Nagel y MAERSK suben. Señal de alerta temprana.
- **Limitación:** DHL no tiene reseñas en el dataset tras jun-2021, por lo que todo el análisis se hace en la ventana común.

---
## 2. Sentimiento en pros y cons (control de calidad)

Uso `distilbert-base-uncased-finetuned-sst-2-english`: modelo transformer preentrenado, binario (POSITIVE/NEGATIVE), rápido y el sugerido en clase. **Por qué binario sirve aquí:** no queremos medir matices, sino comprobar que los textos de *pros* son mayoritariamente positivos y los de *cons* negativos. Si no lo fueran, habría un problema de limpieza o de datos que contaminaría los topics.

`truncation=True` corta en 512 tokens (las reseñas largas son pocas y su tono suele estar claro al principio).

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print('GPU disponible:', torch.cuda.is_available())

clf = pipeline('sentiment-analysis',
               model='distilbert-base-uncased-finetuned-sst-2-english',
               device=device, truncation=True, max_length=512)

def predecir(textos, batch_size=64):
    out = clf(list(textos), batch_size=batch_size)
    return [o['label'] for o in out], [o['score'] for o in out]

# En Colab con GPU tarda ~2-4 min para ~16.000 textos
df['sent_pros'], df['score_pros'] = predecir(df['pros'])
df['sent_cons'], df['score_cons'] = predecir(df['cons'])
df.to_parquet('df_sentimiento.parquet')   # checkpoint para no recalcular

In [ ]:
calidad = (df.groupby('firm')
    .agg(pct_pros_positivos=('sent_pros', lambda s: (s == 'POSITIVE').mean() * 100),
         pct_cons_negativos=('sent_cons', lambda s: (s == 'NEGATIVE').mean() * 100))
    .round(1).loc[orden])
calidad.loc['TOTAL'] = [(df.sent_pros == 'POSITIVE').mean() * 100, (df.sent_cons == 'NEGATIVE').mean() * 100]
calidad.round(1)

In [ ]:
ax = calidad.drop('TOTAL').plot(kind='bar', figsize=(10, 4), color=['#2a9d8f', '#D40511'])
ax.axhline(80, ls='--', c='grey'); ax.set_ylim(0, 100)
ax.set_title('Control de calidad: % pros positivos y % cons negativos'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

### Casos discordantes

In [ ]:
pd.set_option('display.max_colwidth', 200)
cons_pos = df[df['sent_cons'] == 'POSITIVE']
pros_neg = df[df['sent_pros'] == 'NEGATIVE']
print(f'Cons clasificados como POSITIVOS: {len(cons_pos):,} ({len(cons_pos)/len(df):.1%})')
print(f'Pros clasificados como NEGATIVOS: {len(pros_neg):,} ({len(pros_neg)/len(df):.1%})')

print('\n--- Cons POSITIVOS (muestra) ---')
display(cons_pos.sample(10, random_state=SEED)[['firm', 'cons', 'score_cons']])
print('\n--- Pros NEGATIVOS (muestra) ---')
display(pros_neg.sample(10, random_state=SEED)[['firm', 'pros', 'score_pros']])

In [ ]:
# Tipología de discordantes con reglas sencillas, para cuantificar lo que vemos al leerlos
patron_nada = r'^(none|nothing|no cons|n/?a|na|nil|not really|can\'?t think|cannot think|nothing really|no complaints)'
cons_pos = cons_pos.assign(
    tipo=np.select(
        [cons_pos['cons'].str.lower().str.match(patron_nada),
         cons_pos['cons'].str.split().str.len() <= 3],
        ['"No hay cons" / sin quejas', 'Texto muy corto (sin contexto)'],
        'Queja formulada en positivo / mixta'))
cons_pos['tipo'].value_counts(normalize=True).mul(100).round(1)

**Interpretación (completar tras ejecutar):** la gran mayoría de los discordantes no son errores del modelo sino reseñas en las que el empleado escribe *"nothing"*, *"none"*, *"can't think of any"* en cons (el modelo lo lee como positivo), quejas formuladas con palabras amables (*"good company but low pay"*) o pros sarcásticos (*"plenty of overtime, no life"*).

**Decisión para el topic modeling:** los cons del tipo *"no hay cons"* no aportan un tema de queja → se excluyen del modelo de topics de cons (se cuentan aparte como señal positiva).

In [ ]:
# Marcamos los cons vacíos de contenido para excluirlos del topic modeling de cons
df['cons_sin_queja'] = df['cons'].str.lower().str.match(patron_nada)
print(df.groupby('firm')['cons_sin_queja'].mean().mul(100).round(1).loc[orden])